# 进阶实践 1：文件读写与成绩报告

> **运行位置**：短实验在在线 C17 内核中运行；下载的完整程序与编译命令在本地终端运行。在线 `scanf`、`getchar`、`fgets(..., stdin)` 仍不能交互读取键盘。文件流与标准输入不是同一个对象。

先学完前六章，再完成本专题。先预测、运行、修改，最后解释；修改函数或类型定义后重启内核并从头运行。

**学习目标**：区分文件路径与文件流；使用返回值控制读取；检查打开、读写和关闭错误；让程序把统计结果保存到独立报告。

本专题延续 Alice=85、Bob=90、Cathy=58 的数据，文件只存成绩，每行一人。

## 1. 文件名、路径与 FILE 指针

`fopen` 根据路径和模式打开文件，成功返回 `FILE *`，失败返回 `NULL`。`FILE *` 指向库管理的流对象，不是文件内容的字符数组；关闭后不能继续读写。

相对路径从程序的**当前工作目录**解析，不一定是源码所在目录。把可执行文件搬到别处，不会自动搬动数据文件。

| 模式 | 文件不存在时 | 文件存在时 |
| --- | --- | --- |
| `r` | 打开失败 | 从头读取 |
| `w` | 创建 | 清空原内容后写入 |
| `a` | 创建 | 在末尾追加 |
| `wx`（C17 可用） | 创建 | 打开失败，不覆盖已有文件 |

在线实验使用专用文件名 course-file-demo.txt，可以重复覆盖自己的实验数据。这是内核可见的文件，不等于你电脑里的同名文件；保存 Notebook 不等于备份实验生成的文件。需要长期保存的成绩报告请使用后面的本地程序。

## 2. 在线小实验：写入三行

预测文件里会有几行。`fprintf` 与 `printf` 使用类似的格式，但第一个参数指定目标文件流。

先检查 fopen，再检查写入，最后即使写入失败也要尝试关闭。fclose 可能在刷新缓冲数据时报告错误，不能只检查 fprintf。

In [ ]:
#include <stdio.h>
{
    FILE *file = fopen("course-file-demo.txt", "w");
    if (file == NULL) { printf("创建实验文件失败\n"); }
    else {
        int failed = fprintf(file, "85\n90\n58\n") < 0;
        if (fclose(file) != 0) { failed = 1; }
        printf("%s\n", failed ? "写入或关闭失败" : "已写入三行");
    }
}

## 3. 在线小实验：读到结束

先预测每次 fgets 获得的字符串；正常情况下它保留读到的换行符。读取条件直接使用 fgets 的返回值；只有成功读取后才能处理 line。

**修改**：把写入内容换成空字符串，再运行写入与读取单元；应该读取0行。把最后一行的换行删掉，仍应读取3行，但“读取行数”会紧接着58显示。这说明数据与显示格式也有约定。

In [ ]:
{
    FILE *file = fopen("course-file-demo.txt", "r");
    if (file == NULL) { printf("打开实验文件失败\n"); }
    else {
        char line[16];
        size_t count = 0;
        while (fgets(line, sizeof line, file) != NULL) {
            ++count;
            printf("%s", line);
        }
        int failed = ferror(file) != 0;
        if (fclose(file) != 0) { failed = 1; }
        if (failed) { printf("读取或关闭失败\n"); }
        else { printf("读取行数=%zu\n", count); }
    }
}

### EOF 不是提前知道的结束条件

`feof` 反映某次读取是否已遇到文件结束，不能提前预测下一次读取。因此不要写 `while (!feof(file))` 后无条件处理缓冲区。

fgets 返回 NULL 后，先用 ferror 区分读取错误与正常结束。缓冲区太小时，一整行可能分多次读取；上面的短实验只适用于给定短行，不能把每次 fgets 都普遍当成一条完整记录。

后面的完整程序把格式约定写清楚：每行一个0..100的整数、最多100人、每行不超过32字节（不含换行）、没有内嵌 NUL 字节。空文件和空白行都报告错误；末行没有换行也可处理。

## 4. 本地实践：从文件生成报告

将这两个文件下载到同一个新目录，保留原文件名：

- [main.c：完整程序](practice/file_io/main.c)
- [scores.txt：三人成绩](practice/file_io/scores.txt)

打开该目录的终端，执行：

```bash
cc -std=c17 -Wall -Wextra -Wpedantic -Werror main.c -o file_report
./file_report scores.txt report.txt
```

Windows 对应执行 `file_report.exe scores.txt report.txt`。终端显示“已生成报告”；打开新生成的 report.txt，应看到：

```text
人数=3
平均分=77.67
最高分=90
及格=2
```

完整程序使用 `wx` 创建报告。如果 report.txt 已存在，程序会明确失败；再次实验请使用新报告名，例如 report2.txt。输入无效时不会创建报告。

`int main(int argc, char *argv[])` 接收命令行参数：本例 argc 应为3，argv[1] 是输入路径，argv[2] 是输出路径。它与通过 scanf 读取键盘是不同的输入方式。

## 5. 读懂完整程序的三个阶段

1. 打开输入，逐行验证并累计；失败时停止处理。
2. 检查读流错误并关闭输入；有错误或没有成绩就结束，不生成部分报告。
3. 创建独立输出文件，写报告并检查关闭结果。

`parse_score` 只读取字符串并返回解析结果；文件操作留在 main 中。`%4d` 限制数字转换最多使用4个字符，数值仍在 C17 保证的 int 范围内；后面的 `%c` 检查有无多余非空白内容。因此 `85abc`、`85 90`、超长数字不会被当成合法单人成绩。输入允许前后空白和 `+100`。

固定宽度是本题0..100格式的设计，不能照搬为任意整数解析器。理解格式约定，比仅判断“转换成功”更重要。

## 6. 分层练习与验收

### A. 预测与观察

把 scores.txt 改为一行60，且末尾没有换行。写下报告的四行，再运行核对。

<details><summary>提示与答案</summary>

人数1、平均60.00、最高60、及格1；成功读取末行不要求它带换行。

</details>

### B. 自己制造一次可解释的错误

每次只改一项：文件不存在、空文件、第二行 abc、成绩101、同一行两个成绩。记录退出状态和错误信息；检查是否错误地生成了报告。

验收：均失败，不创建新报告；第二行 abc 应指出第2行。

### C. 添加不及格人数

修改自己的 main.c，使报告多一行“不及格=…”。验收：默认数据为1；`60,100` 两行为0；单人成绩0为1。已有的及格人数和平均分不能改变。

<details><summary>一级提示</summary>

只统计合法成绩时，总人数与及格人数已足够计算不及格人数。

</details>

<details><summary>二级提示</summary>

输出 count - passed，格式使用 %zu；不要额外维护会与既有结果不一致的计数器。

</details>

### D. 边界挑战

生成100行60应成功，101行60应失败。将输出路径改成输入路径，或使用已有报告名，文件内容应保持不变。解释 wx 模式与 w 模式的差别。

写操作失败可能留下不完整的新报告，因此成功条件必须包含写入和关闭成功；“看到了文件”本身不是验收通过。

In [ ]:
// 学习记录：写下一个错误案例、程序报告的位置，以及你如何修复。

## 小结

文件处理需要同时考虑数据格式、读取返回值和资源释放。把“读取成功”“数据合法”“报告写入成功”分开判断，才能解释程序在哪一步失败。下一专题将统计规则拆到独立模块。